In [7]:
from glob import glob
import pandas as pd
ROOT_DIR = "/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data"
all_files = glob(ROOT_DIR+"/train_sft*")
print(len(all_files), all_files)
from tqdm import tqdm
import pandas as pd
import pyarrow.parquet as pq
import json
import re
import random

3 ['/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00000-of-00003-a3ecf92756993583.parquet', '/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00002-of-00003-ee46ed25cfae92c6.parquet', '/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00001-of-00003-0a1804bcb6ae68c6.parquet']


In [8]:
all_files

['/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00000-of-00003-a3ecf92756993583.parquet',
 '/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00002-of-00003-ee46ed25cfae92c6.parquet',
 '/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00001-of-00003-0a1804bcb6ae68c6.parquet']

In [9]:
from transformers import AutoTokenizer
model_path = "/DATA/disk2/yuhang/.cache/modelscope/models/Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

In [18]:
# 读取第一个parquet文件并查看一条样本数据
file_path = "/DATA/disk2/yuhang/.cache/modelscope/datasets/swift/ultrachat_200k/data/train_sft-00000-of-00003-a3ecf92756993583.parquet"

# 使用pyarrow读取parquet文件
table = pq.read_table(file_path)
df = table.to_pandas()

# 打印数据集的基本信息
print("数据集形状:", df.shape)
print("列名:", df.columns.tolist())
print("\n" + "="*50)

# 查看第一条样本数据
print("第一条样本数据:")
sample = df.iloc[0]
print(sample)
print("Prompt ID:", sample['prompt_id'])
print("Prompt:", sample['prompt'])

messages = sample['messages']
print("对话轮数:", len(messages))

# 打印对话的详细内容
for i, turn in enumerate(messages):
    role = turn['from'] if 'from' in turn else turn.get('role', '未知角色')
    content = turn['value'] if 'value' in turn else turn.get('content', '无内容')
    print(f"第{i+1}轮 - {role}: {content[:200]}...")  # 只显示前200个字符
    print("-" * 30)


数据集形状: (69289, 3)
列名: ['prompt', 'prompt_id', 'messages']

第一条样本数据:
prompt       These instructions apply to section-based them...
prompt_id    f0e37e9f7800261167ce91143f98f511f768847236f133...
messages     [{'content': 'These instructions apply to sect...
Name: 0, dtype: object
Prompt ID: f0e37e9f7800261167ce91143f98f511f768847236f133f2d0aed60b444ebe57
Prompt: These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!
Your Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.
Does this feature apply to all sections of the theme or just specific ones as listed in the text material?
对话轮数: 8
第1轮 - user: These instructions apply to section

In [ ]:
# def to_share_gpt_data(messages):
#     """
#     messages: List[dict], 例如 [{"role":"user","content":"..."}, ...]
#     返回: dict, 例如 {"conversations": [{"role":..., "content":...}, ...]}
#     """
#     conversations = []
#     text = ""
#     for item in messages:
#         # 确保有role和content字段
#         conversations.append({
#             "role": item["role"],
#             "content": item["content"]
#         })
#         text += item["content"]
#     return {
#         "conversations": conversations
#     }, text


In [25]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
# from transformers import AutoTokenizer # 假设您将使用 tokenizer
# from tqdm import tqdm # 用于显示进度条

# 假设 tokenizer 已经加载，并且它包含一个聊天模板
# 例如: tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

# to_share_gpt_data 函数保持不变，它已经很好了。
def to_share_gpt_data(messages_data):
    # ... (与您提供的版本相同)
    conversations = []
    role_mapping = {"user": "human", "assistant": "gpt"}
    for item in messages_data:
        if "role" not in item or "content" not in item:
            continue
        role = item["role"]
        content = item["content"]
        from_value = role_mapping.get(role, role)
        conversations.append({"from": from_value, "value": content})
    return {"conversations": conversations}

def process_one_data(file_dir, tokenizer):
    """
    处理单个 Parquet 文件，进行健壮性检查，并转换为 ShareGPT 格式。
    同时，使用模型的聊天模板生成用于计算 token 长度的文本。

    Args:
        file_dir (str): Parquet 文件的路径。
        tokenizer: 已经加载的 Hugging Face tokenizer 实例。

    Returns:
        tuple: 包含两个列表 (处理好的 ShareGPT 格式数据, 用于计算 token 的文本列表)。
    """
    try:
        table = pq.read_table(file_dir)
        df = table.to_pandas()
    except Exception as e:
        print(f"Error reading or processing file {file_dir}: {e}")
        return [], []

    # 关键：确定包含对话列表的列名，这里假设是 'messages'
    # 如果是其他名称，如 'conversations', 请在此处修改
    if "messages" not in df.columns:
        print(f"Column 'messages' not found in {file_dir}. Skipping.")
        return [], []
    
    source_conversations = df["messages"]

    valid_datas = []
    texts_for_tokenization = []
    
    # 初始化统计计数器
    stats = {
        "total": len(source_conversations),
        "skipped_empty": 0,
        "skipped_odd_turns": 0,
        "skipped_starts_with_assistant": 0,
        "processed_ok": 0
    }

    # === 合并循环，提高效率 ===
    for messages_list in source_conversations:
        # # 检查是否为 NaN (Pandas的空值)
        # if pd.isna(messages_list):
        #     stats["skipped_empty_or_na"] += 1
        #     continue

        # # 检查是否为 NumPy 数组，如果是，则转换为 Python 列表
        # if isinstance(messages_list, np.ndarray):
        #     messages_list = messages_list.tolist()
        # # === 核心修正点 END ===

        # # 现在 messages_list 可以保证是 list 或者其他非数组类型
        # # 1. 健壮性检查：过滤空列表或非列表数据
        # if not isinstance(messages_list, list) or len(messages_list) == 0:
        #     stats["skipped_empty_or_na"] += 1
        #     continue
            

        # 3. 健壮性检查：确保对话由用户发起
        if messages_list[0].get("role") != "user":
            stats["skipped_starts_with_assistant"] += 1
            continue
            
        # 4. 核心转换 (您的原始 messages 格式 -> ShareGPT 的 from/value 格式)
        # 注意：to_share_gpt_data 内部的 role 已经映射为 from
        share_gpt_item = to_share_gpt_data(messages_list)
        
        # 5. 使用聊天模板格式化文本 (关键步骤！)
        # tokenizer.apply_chat_template 需要一个 [{"role": "user", "content": "..."}, ...] 格式的列表
        # 所以我们直接使用原始的 messages_list
        # add_generation_prompt=False 表示我们已有完整对话，不需要再添加一个助手的提示
        try:
            formatted_text = tokenizer.apply_chat_template(
                messages_list, 
                tokenize=False, 
                add_generation_prompt=False
            )
        except Exception as e:
            # 如果某个对话无法应用模板，跳过它
            print(f"Could not apply chat template to an item. Error: {e}")
            continue

        valid_datas.append(share_gpt_item)
        texts_for_tokenization.append(formatted_text)
        stats["processed_ok"] += 1

    print(f"File: {file_dir.split('/')[-1]} | "
          f"Total: {stats['total']} | "
          f"Processed: {stats['processed_ok']} | "
          f"Skipped (Empty: {stats['skipped_empty']}, Odd Turns: {stats['skipped_odd_turns']}, Not starting with user: {stats['skipped_starts_with_assistant']})")

    return valid_datas, texts_for_tokenization

# --- 主执行逻辑 (修改后示例) ---
all_processed_datas = []
all_token_nums = []

for file_dir in tqdm(all_files, desc="Processing files"):
    # 将 tokenizer 传入处理函数
    datas, texts = process_one_data(file_dir, tokenizer) 
    
    if datas:
        all_processed_datas.extend(datas)
        # 直接对返回的、已正确格式化的文本进行分词
        input_ids = tokenizer(texts)['input_ids']
        all_token_nums.extend([len(x) for x in input_ids])

print("\n--- Overall Stats ---")
print(f"Total valid conversations processed: {len(all_processed_datas)}")
print(f"Total token count list length: {len(all_token_nums)}")
if all_token_nums:
    print(f"Average token count: {sum(all_token_nums) / len(all_token_nums):.2f}")

Processing files:   0%|          | 0/3 [00:00<?, ?it/s]

File: train_sft-00000-of-00003-a3ecf92756993583.parquet | Total: 69289 | Processed: 69289 | Skipped (Empty: 0, Odd Turns: 0, Not starting with user: 0)


Processing files:  33%|███▎      | 1/3 [01:09<02:19, 69.89s/it]

File: train_sft-00002-of-00003-ee46ed25cfae92c6.parquet | Total: 69288 | Processed: 69288 | Skipped (Empty: 0, Odd Turns: 0, Not starting with user: 0)


Processing files:  67%|██████▋   | 2/3 [02:33<01:18, 78.25s/it]

File: train_sft-00001-of-00003-0a1804bcb6ae68c6.parquet | Total: 69288 | Processed: 69288 | Skipped (Empty: 0, Odd Turns: 0, Not starting with user: 0)


Processing files: 100%|██████████| 3/3 [03:51<00:00, 77.25s/it]


--- Overall Stats ---
Total valid conversations processed: 207865
Total token count list length: 207865
Average token count: 1189.44


In [27]:
all_processed_datas[0]

{'conversations': [{'from': 'human',
   'value': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?"},
  {'from': 'gpt',
   'value': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.'},
  {'from': 'human',
   'value': 'Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?'},
  {'from': 'gpt

In [28]:
import os
file_path = '/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input/ultrachat_200k.jsonl'

directory = os.path.dirname(file_path)

if not os.path.exists(directory):
    os.makedirs(directory)
    print(f"目录{directory} 不存在，已创建")
else:
    print(f"目录{directory} 已存在")

with open(file_path, 'w', encoding="utf-8") as f:
    for item in all_processed_datas:
        json.dump(item, f, ensure_ascii=False)
        f.write('\n')

目录/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input 已存在
